# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [12]:
"""
Your Turn - Production-Ready Experiment
"""

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import precision_score, recall_score
from sklearn.inspection import permutation_importance

# 1. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 2. Find optimal depth with cross-validation
depths = range(2, 8)
cv_results = []
best_depth = 2
best_cv_score = 0

print("Depth | CV Precision (mean±std) | Test Precision@K")
print("-" * 60)

for depth in depths:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        min_samples_split=50,
        random_state=42
    )

    # Cross-validation
    cv_scores = cross_val_score(tree, X_train, y_train, cv=5, scoring='precision')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    # Train final model for this depth
    tree.fit(X_train, y_train)
    scores = tree.predict_proba(X_test)[:, 1]

    p20 = precision_at_k(scores, y_test, 20)
    p50 = precision_at_k(scores, y_test, 50)
    p100 = precision_at_k(scores, y_test, 100)

    marker = "← best" if depth == best_depth else ""
    print(f"  {depth}   |  {cv_mean:.3f} (±{cv_std:.3f})      |  {p20:.3f}@{20}  {p50:.3f}@{50}  {p100:.3f}@{100}  {marker}")

    cv_results.append({
        'depth': depth,
        'cv_mean': cv_mean,
        'cv_std': cv_std,
        'p20': p20,
        'p50': p50,
        'p100': p100,
        'tree': tree
    })

    # Track best by CV score
    if cv_mean > best_cv_score:
        best_cv_score = cv_mean
        best_depth = depth
        best_tree = tree

# 3. Get best tree results
best_result = [r for r in cv_results if r['depth'] == best_depth][0]

print(f"\n✅ Best depth: {best_depth} (CV: {best_cv_score:.3f})")

# 4. Show the tree (first 3 levels for readability)
print(f"\nBest Tree (depth {best_depth}):")
print(export_text(best_tree, feature_names=features, max_depth=3))

# 5. Feature importance (permutation)
perm_importance = permutation_importance(
    best_tree, X_test, y_test,
    n_repeats=10,
    random_state=42,
    scoring='precision'
)

print("\nFeature Importance (Permutation):")
for idx in perm_importance.importances_mean.argsort()[::-1]:
    mean = perm_importance.importances_mean[idx]
    std = perm_importance.importances_std[idx]
    print(f"  {features[idx]:25s}: {mean:.4f} (±{std:.4f})")

# 6. Business impact
hand_scores = df.loc[X_test.index, "hand_rule_score"].values
tree_scores = best_tree.predict_proba(X_test)[:, 1]

print("\nBusiness Impact:")
print("-" * 40)
for k in [20, 50, 100]:
    hand = precision_at_k(hand_scores, y_test, k)
    tree = precision_at_k(tree_scores, y_test, k)
    improvement = ((tree - hand) / hand * 100) if hand > 0 else 0
    print(f"  @{k}: Hand={hand:.3f} → Tree={tree:.3f} (+{improvement:.1f}%)")

# 7. Production checks
print("\nProduction Check:")
print("-" * 40)

low_imp = X_test[X_test['impressions_90d'] < 100]
if len(low_imp) > 0:
    low_score = precision_at_k(
        best_tree.predict_proba(low_imp)[:, 1],
        y_test[X_test['impressions_90d'] < 100],
        50
    )
    print(f"  Low impressions (<100): {low_score:.3f}")

stale = X_test[X_test['days_since_last_update'] > 365]
if len(stale) > 0:
    stale_score = precision_at_k(
        best_tree.predict_proba(stale)[:, 1],
        y_test[X_test['days_since_last_update'] > 365],
        50
    )
    print(f"  Stale pages (>1yr): {stale_score:.3f}")

# 8. Simplicity check
print("\nSimplicity Check:")
print("-" * 40)

simple_tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)
simple_tree.fit(X_train, y_train)

simple_score = precision_at_k(
    simple_tree.predict_proba(X_test)[:, 1],
    y_test,
    50
)

best_p50 = best_result['p50']
perf_loss = best_p50 - simple_score

print(f"  Depth {best_depth} score: {best_p50:.3f}")
print(f"  Depth 2 score: {simple_score:.3f}")
print(f"  Loss: {perf_loss:.3f} ({perf_loss/best_p50*100:.1f}%)")

# 9. Final recommendation
print("\n" + "="*50)
print("RECOMMENDATION")
print("="*50)

if perf_loss < 0.02:
    print(f"✅ Use depth 2 - simpler, readable, nearly same performance")
    print(f"   Depth 2 tree is fully readable and business can understand it")
else:
    print(f"✅ Use depth {best_depth} - better performance, still interpretable")
    print(f"   {perf_loss*100:.1f}% gain over depth 2")

print(f"\nDeploy depth {2 if perf_loss < 0.02 else best_depth} with monitoring")

Depth | CV Precision (mean±std) | Test Precision@K
------------------------------------------------------------
  2   |  0.641 (±0.002)      |  0.650@20  0.680@50  0.640@100  ← best
  3   |  0.653 (±0.015)      |  0.600@20  0.600@50  0.630@100  
  4   |  0.663 (±0.006)      |  0.850@20  0.780@50  0.740@100  
  5   |  0.676 (±0.010)      |  0.950@20  0.840@50  0.820@100  
  6   |  0.677 (±0.008)      |  0.850@20  0.780@50  0.810@100  
  7   |  0.677 (±0.007)      |  0.800@20  0.860@50  0.810@100  

✅ Best depth: 6 (CV: 0.677)

Best Tree (depth 6):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.65
|   |   |--- word_count <= 687.00
|   |   |   |--- class: 0
|   |   |--- word_count >  687.00
|   |   |   |--- days_since_last_update <= 3.50
|   |   |   |   |--- class: 0
|   |   |   |--- days_since_last_update >  3.50
|   |   |   |   |--- truncated branch of depth 3
|   |--- avg_position >  0.65
|   |   |--- content_age_days <= 108.50
|   |   |   |--- content_age_days <= 105.50
|   |

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.